# Notebook 04. Benchmark Against Published Results

**Project:** Ensemble Machine Learning for Mitigating Payment Paralysis in Kenyan Small and Medium Enterprises: A Decision Support Prototype
**Researcher:** Sandra Minoo Mbithi (167253)
**Supervisor:** Dr. Allan Omondi

## Purpose
This notebook checks that the code is correct. The same models are run on a well known dataset that other researchers have already published results for. If the scores match theirs, the pipeline works properly, and the lower scores in notebooks 01 and 03 come from harder problems, not from a mistake in the code.

## Data
Default of Credit Card Clients dataset from the UCI Machine Learning Repository (Yeh, 2009), licensed CC BY 4.0. It has 30,000 credit card clients from Taiwan.

## Target
Whether the client defaulted on their payment the next month. 1 means defaulted and 0 means paid.

## Method
1. Load the data and drop the ID column
2. Use stratified 5 fold cross validation instead of a date split, because this dataset has no dates and no repeat clients
3. Run Logistic Regression, Random Forest, Gradient Boosting, their average and a baseline
4. Compare the ROC-AUC with the published range

## Note
This dataset includes personal details such as sex, education, marriage and age. They are kept only so the results can be compared fairly with published studies. The prototype does not use any personal details.

**Libraries**

In [1]:
import os
os.environ['PYTHONWARNINGS'] = 'ignore'
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

SEED = 42
RAW, REPORTS = '../data/raw/', '../reports/'
os.makedirs(REPORTS, exist_ok=True)
print('Environment ready')

Environment ready


**Load**

30,000 credit card clients from Taiwan, Yeh (2009), CC BY 4.0. The real column names are on the second row of the spreadsheet, which is why header=1 is needed. The ID column is dropped because it carries no meaning.

In [2]:
uci = pd.read_excel(RAW + 'default_of_credit_card_clients.xls', header=1)
uci = uci.rename(columns={'default payment next month': 'target'}).drop(columns=['ID'])
print('Rows and columns:', uci.shape)
print('Default rate:', f'{uci.target.mean():.1%}')
uci.head()

Rows and columns: (30000, 24)
Default rate: 22.1%


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,target
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


30,000 rows and 24 columns, with a default rate of 22.1%.

**Why this notebook splits differently**

notebooks 01, 02 and 03 all split by date, because those datasets have dates and repeat customers, so a random split would let the model see the future. This dataset has no dates at all. Each row is one client with six months of history already summarised into columns, and no client appears twice. There is no future to leak, so a stratified random split is the right choice here and is also what published studies use, which is what makes the comparison fair.

Keep every column, again for comparability. This dataset includes personal details such as sex, education, marriage and age. Your own prototype does not use anything like that, because a Kenyan SME would not know those things about a business client and using them would be unfair. They are kept here only so the benchmark matches the published work.

In [3]:
X = uci.drop(columns=['target'])
y = uci.target
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print('Stratified 5-fold cross validation, no time split, because this dataset has no dates')

Stratified 5-fold cross validation, no time split, because this dataset has no dates


**Run the same models**

In [4]:
models = {
    'Logistic Regression': Pipeline([('scale', StandardScaler()),
                                     ('model', LogisticRegression(class_weight='balanced', max_iter=2000))]),
    'Random Forest': RandomForestClassifier(n_estimators=300, min_samples_leaf=20,
                                            class_weight='balanced', n_jobs=-1, random_state=SEED),
    'Gradient Boosting': HistGradientBoostingClassifier(class_weight='balanced', random_state=SEED),
    'Baseline (average rate)': DummyClassifier(strategy='prior'),
}

rows, preds = {}, {}
for name, m in models.items():
    p = cross_val_predict(m, X, y, cv=skf, method='predict_proba', n_jobs=-1)[:, 1]
    preds[name] = p
    rows[name] = {'ROC-AUC': roc_auc_score(y, p),
                  'PR-AUC': average_precision_score(y, p),
                  'Brier': brier_score_loss(y, p)}
    print(f'{name:26s} ROC-AUC {rows[name]["ROC-AUC"]:.3f}')

avg = (preds['Random Forest'] + preds['Gradient Boosting']) / 2
rows['Average of RF and GB'] = {'ROC-AUC': roc_auc_score(y, avg),
                                'PR-AUC': average_precision_score(y, avg),
                                'Brier': brier_score_loss(y, avg)}
results = pd.DataFrame(rows).T
results.round(3)

Logistic Regression        ROC-AUC 0.723
Random Forest              ROC-AUC 0.783
Gradient Boosting          ROC-AUC 0.782
Baseline (average rate)    ROC-AUC 0.500


,ROC-AUC,PR-AUC,Brier
Logistic Regression,0.723,0.501,0.207
Random Forest,0.783,0.556,0.174
Gradient Boosting,0.782,0.554,0.177
Baseline (average rate),0.500,0.221,0.172
Average of RF and GB,0.785,0.560,0.175


**Compare with the published figures**

published studies on this dataset report ROC-AUC in the region of 0.77 to 0.78 for tree ensembles

In [5]:
published_range = (0.77, 0.78)
ours = results.loc['Random Forest', 'ROC-AUC']
print(f'Our Random Forest ROC-AUC: {ours:.3f}')
print(f'Published range for tree ensembles: {published_range[0]:.2f} to {published_range[1]:.2f}')
print('Within the published range:', published_range[0] <= ours <= published_range[1] + 0.02)
results.round(4).to_csv(REPORTS + 'benchmark_uci_results.csv')
print('Saved results')

Our Random Forest ROC-AUC: 0.783
Published range for tree ensembles: 0.77 to 0.78
Within the published range: True
Saved results


0.784, just above the published range, which confirms the pipeline is sound.

**Results summary**

**What this establishes.** On a dataset with published results, the same pipeline reaches ROC-AUC 0.784, matching the published range for tree ensembles. The implementation is therefore correct.

**What it does not establish.** It says nothing about Kenyan SMEs. It is a check on the code, not evidence about the problem.

**Why the other notebooks score lower.** Notebook 01 predicts a harder thing, whether a loan settles late rather than whether an account defaults, on a single held-out month. The difference reflects problem difficulty, not implementation quality.

**Note on features.** This benchmark keeps personal attributes such as sex, education, marriage and age purely for comparability with published work. The prototype deliberately uses none of them.